In [ ]:


def decode_hls_fmask(fmask):
    """
    Decode the 8-bit Fmask layer from HLS (HLSL30 or HLSS30) into individual masks.

    Parameters
    ----------
    fmask : numpy.ndarray or xarray.DataArray
        The 8-bit Fmask array (values 0–255).

    Returns
    -------
    dict
        Dictionary containing boolean masks for:
        - water
        - snow_ice
        - cloud_shadow
        - adjacent_shadow
        - cloud
    """

    # Ensure we’re working with a NumPy array
    fmask = fmask.values if isinstance(fmask, xr.DataArray) else fmask

    # Convert to uint8 for bitwise ops
    fmask = fmask.astype(np.uint8)


    # --- Bit extraction ---
    # Bits 0–1: Cloud confidence
    cloud_bit = (fmask >> 1) & 0b1

    # Bits 1–2: Cloud Adjencent
    adj_cloud_bit = (fmask >> 2) & 0b1

    # Bits 2–3: Cloud shadow confidence
    cloud_shadow_bit = (fmask >> 3) & 0b1

    # Bits 4–5: Snow/Ice confidence
    snow_bit = (fmask >> 4) & 0b1

    # Bits 6–7: Water confidence
    water_bit = (fmask >> 5) & 0b1

    # Bits 8-: Aerosol confidence
    aerosol_bits = (fmask >> 6) & 0b11

    masks = {
        "aerosol": aerosol_bits,  # 0-3 (level)
        "water": water_bit == 1,
        "snow_ice": snow_bit == 1,
        "cloud_shadow": cloud_shadow_bit == 1,
        "adjacent_cloud_shadow": adj_cloud_bit == 1,
        "cloud": cloud_bit == 1
    }

    return masks


def convert_fmask_unique(fmask):
    masks = decode_hls_fmask(fmask)
    classified = np.full(fmask.shape, np.nan)

    new_values = [1,2,3,4]

    # Assign priorities: water < snow/ice < shadow < adjacent_shadow < cloud
    classified[masks["water"]] = 1
    classified[masks["snow_ice"]] = 3
    classified[masks["cloud_shadow"]] = 2
    classified[masks["cloud"]] = 4

    mask_nan = ~np.isin(classified, new_values)
    classified[mask_nan] = np.nan


    return classified



        
def apply_mask(band_data, mask):
    """
        Apply mask to the band data
    """
    masked_raster = band_data.where(mask!=True)       
    return masked_raster

def get_fmask_info(i):
    """
    Return fmask version info and title based on index.

    Parameters:
        i (int): Index (0, 1, or other).
        day (str, optional): Day string to format (e.g. '2025_11_05').

    Returns:
        dict: {
            'fmask_version': str,
            'title': str,
            'day_text': str or None
        }
    """
    if i == 0:
        fmask_version = ''
        title = 'Reference'

    elif i == 1:
        fmask_version = 'fmask4.7'
        title = 'Fmask 4.7'
        
    elif i == 3:
        fmask_version = 'cirrus'
        title = 'Cirrus band TOA'
        
    else:
        fmask_version = 'fmask5_May2025'
        title = 'Fmask 5.0.1'

    return {
        'fmask_version': fmask_version,
        'title': title,
    }


In [ ]:
def get_fmask_percentage_compare(files_s3_fmask5, files_s3_fmask4, days,
                                 bucket_name="hls-debug-output",
                                 output_dir="./",
                                 cache=True,
                                 cache_suffix="",
                                 version1_label="fmask4",
                                 version2_label="fmask5"):
    """
    Calculate Fmask percentage comparison between Fmask 4 and Fmask 5.
    Returns one row per granule with columns for each feature and differences.
    
    Parameters:
    -----------
    files_s3_fmask5 : list
        List of Fmask 5 file paths
    files_s3_fmask4 : list
        List of Fmask 4 file paths
    days : list
        List of days to process
    output_dir : str, optional
        Directory to save/read CSV cache file. Default is './'
    cache : bool, optional
        If True, check for existing CSV and save results. Default is True
    
    Returns:
    --------
    pandas.DataFrame
        DataFrame with one row per granule (Date + mrgs_id) with columns:
        - Date: Date string
        - mrgs_id: MRGS tile ID
        - Water_fmask4: Water percentage for Fmask 4.7
        - Water_fmask5: Water percentage for Fmask 5
        - CloudShadow_fmask4: Cloud shadow percentage for Fmask 4.7
        - CloudShadow_fmask5: Cloud shadow percentage for Fmask 5
        - SnowIce_fmask4: Snow/Ice percentage for Fmask 4.7
        - SnowIce_fmask5: Snow/Ice percentage for Fmask 5
        - Cloud_fmask4: Cloud percentage for Fmask 4.7
        - Cloud_fmask5: Cloud percentage for Fmask 5
        - ValidPixels_fmask4: Count of valid (non-NaN) pixels for Fmask 4.7
        - ValidPixels_fmask5: Count of valid (non-NaN) pixels for Fmask 5
        - Cloud_diff: Difference in cloud percentage (Fmask5 - Fmask4)
        - CloudShadow_diff: Difference in cloud shadow percentage (Fmask5 - Fmask4)
    """
    import os
    import hashlib
    
    # Generate cache filename based on input parameters
    if cache:
        # Create a hash from the days and file lists to create unique cache filename
        cache_key = '_'.join(sorted(days)) + '_' + str(len(files_s3_fmask5)) + '_' + str(len(files_s3_fmask4))
        cache_hash = hashlib.md5(cache_key.encode()).hexdigest()[:8]
        
        # Build cache filename with optional suffix
        suffix_str = f'_{cache_suffix}' if cache_suffix else ''
        csv_filename = os.path.join(output_dir, f'df_fmask_percentage{suffix_str}_{cache_hash}.csv')
        
        # Check if CSV exists
        if os.path.exists(csv_filename):
            print(f'Reading cached results from {csv_filename}')
            df_fmask_percentage = pd.read_csv(csv_filename)
            return df_fmask_percentage
    
    # If CSV doesn't exist, execute the function
    granule_data_list = list()
    
    for day in days:
        # convert date to julian date
        day_julian = day
        
        # get list of mrgs_ids within this certain date from Fmask5 files
        mrgs_ids_fmask5 = np.unique([i.split('.')[2] for i in files_s3_fmask5 if day in i])
        
        # Process each MRGS ID separately
        for mrgs in mrgs_ids_fmask5:
            # Check if we have both fmask4 and fmask5 for this specific MRGS ID and day
            matches_fmask4 = [f for f in files_s3_fmask4 if day_julian in f and mrgs in f and f.endswith('.tif')]
            matches_fmask5 = [f for f in files_s3_fmask5 if day in f and mrgs in f and f.endswith('.tif')]
            
            # Check if we have at least one Fmask file for both versions
            file_fmask4_list = [i for i in matches_fmask4 if 'Fmask.tif' in i]
            file_fmask5_list = [i for i in matches_fmask5 if 'Fmask' in i or 'Fmask.tif' in i]
            
            if len(file_fmask4_list) == 0 or len(file_fmask5_list) == 0:
                print(f'No matching Fmask files for MRGS ID {mrgs} on date {day} (Fmask4: {len(file_fmask4_list)}, Fmask5: {len(file_fmask5_list)})')
                continue
            
            # Use the first matching file for each version
            file_fmask4 = file_fmask4_list[0]
            file_fmask5 = file_fmask5_list[0]
            
            # Initialize dictionary for this granule
            granule_data = {
                'Date': day,
                'mrgs_id': mrgs
            }
            
            # Process both Fmask versions
            for file, version_suffix in [(file_fmask4, version1_label), (file_fmask5, version2_label)]:
                fmask_version = file.split('_')[0]
                
                raster = rasterio.open('s3://'+bucket_name+'/'+file).read(1).astype(np.float16)

                # Count valid pixels before conversion (exclude 255 fill values)
                valid_pixel_count = np.count_nonzero((raster != 255))
                granule_data[f'ValidPixels_{version_suffix}'] = valid_pixel_count
                
                # Convert to unique classes for comparison
                raster = convert_fmask_unique(raster)
                
                # mask out 255 to nan
                raster[raster == 255] = np.nan
                raster[raster == 0] = np.nan #ignore the land classification
                
                # Calculate percentage for each feature
                # Classes: 1=Water, 2=Cloud Shadow, 3=Snow/Ice, 4=Cloud
                classes_all = ['Land/NaN','Water','Cloud Shadow','Snow/Ice','Cloud']
                total_pixels = raster.shape[0] * raster.shape[1]
                
                for code in [1, 2, 3, 4]:  # Water, Cloud Shadow, Snow/Ice, Cloud
                    raster_copy = raster.copy()
                    raster_copy[raster_copy == code] = 10
                    raster_copy[raster_copy != 10] = 0
                    
                    # Calculate percentage
                    percentage = np.count_nonzero(raster_copy) / total_pixels * 100
                    
                    # Store in dictionary with appropriate column name
                    feature_name = classes_all[code]
                    if feature_name == 'Water':
                        granule_data[f'Water_{version_suffix}'] = percentage
                    elif feature_name == 'Cloud Shadow':
                        granule_data[f'CloudShadow_{version_suffix}'] = percentage
                    elif feature_name == 'Snow/Ice':
                        granule_data[f'SnowIce_{version_suffix}'] = percentage
                    elif feature_name == 'Cloud':
                        granule_data[f'Cloud_{version_suffix}'] = percentage
            
            # Calculate differences (Fmask5 - Fmask4)
            if 'Cloud_fmask4' in granule_data and 'Cloud_fmask5' in granule_data:
                granule_data['Cloud_diff'] = granule_data['Cloud_fmask5'] - granule_data['Cloud_fmask4']
            else:
                granule_data['Cloud_diff'] = np.nan
                
            if 'CloudShadow_fmask4' in granule_data and 'CloudShadow_fmask5' in granule_data:
                granule_data['CloudShadow_diff'] = granule_data['CloudShadow_fmask5'] - granule_data['CloudShadow_fmask4']
            else:
                granule_data['CloudShadow_diff'] = np.nan
            
            granule_data_list.append(granule_data)
    
    # Convert to DataFrame
    df_fmask_percentage = pd.DataFrame(granule_data_list)
    
    # Ensure all expected columns exist (fill with 0 if missing)
    expected_cols = ['Date', 'mrgs_id', 
                     'Water_fmask4', 'Water_fmask5',
                     'CloudShadow_fmask4', 'CloudShadow_fmask5',
                     'SnowIce_fmask4', 'SnowIce_fmask5',
                     'Cloud_fmask4', 'Cloud_fmask5',
                     'ValidPixels_fmask4', 'ValidPixels_fmask5',
                     'Cloud_diff', 'CloudShadow_diff']
    
    for col in expected_cols:
        if col not in df_fmask_percentage.columns:
            if col in ['Date', 'mrgs_id']:
                continue  # These should always exist
            df_fmask_percentage[col] = 0.0
    
    # Reorder columns
    df_fmask_percentage = df_fmask_percentage[expected_cols]
    
    # Save to CSV if cache is enabled
    if cache:
        # Create output directory if it doesn't exist
        os.makedirs(output_dir, exist_ok=True)
        df_fmask_percentage.to_csv(csv_filename, index=False)
        print(f'Results saved to {csv_filename}')

    return df_fmask_percentage

In [ ]:
def calculate_cloud_coverage_timeseries(fmask4_data, fmask5_data, verbose=True, mask_value=4):
    """
    Calculate coverage percentage for a specific mask value for each time step in Fmask 4.7 and 5.0 datasets.
    
    Can be used for cloud (mask_value=4) or cloud shadow (mask_value=2).
    
    Parameters
    ----------
    fmask4_data : xarray.DataArray or None
        Fmask 4.7 time-series DataArray with dimensions (time, y, x).
        Values: 1=water, 2=cloud_shadow, 3=snow_ice, 4=cloud, NaN=clear.
    fmask5_data : xarray.DataArray or None
        Fmask 5.0 time-series DataArray with dimensions (time, y, x).
        Values: 1=water, 2=cloud_shadow, 3=snow_ice, 4=cloud, NaN=clear.
    verbose : bool, optional
        If True, print the comparison results. Default is True.
    mask_value : int, optional
        Fmask value to calculate coverage for. Default is 4 (cloud).
        Use 2 for cloud shadow, 4 for cloud.
    
    Returns
    -------
    pandas.DataFrame
        DataFrame with columns:
        - time: Time coordinate for each observation
        - fmask4_coverage: Coverage percentage for Fmask 4.7
        - fmask5_coverage: Coverage percentage for Fmask 5.0
        - difference: Difference (Fmask 5.0 - 4.7) in percentage points
    """
    import pandas as pd
    import numpy as np
    
    if fmask4_data is None or fmask5_data is None:
        if verbose:
            print("Warning: One or both Fmask datasets are None. Cannot calculate coverage.")
        return None
    
    # Find common time steps
    common_times = sorted(set(fmask4_data.time.values) & set(fmask5_data.time.values))
    
    if not common_times:
        if verbose:
            print("Warning: No common time steps between Fmask 4.7 and 5.0.")
        return None
    
    results = []
    
    for time_val in common_times:
        fmask4_slice = fmask4_data.sel(time=time_val)
        fmask5_slice = fmask5_data.sel(time=time_val)
        
        # Count pixels for the specified mask_value
        count4 = (fmask4_slice == mask_value).sum().values
        count5 = (fmask5_slice == mask_value).sum().values
        
        # Count total valid (non-NaN) pixels
        valid4_count = (~np.isnan(fmask4_slice)).sum().values
        valid5_count = (~np.isnan(fmask5_slice)).sum().values
        
        # Calculate coverage percentage
        if valid4_count > 0:
            coverage4 = (count4 / valid4_count) * 100.0
        else:
            coverage4 = np.nan
        
        if valid5_count > 0:
            coverage5 = (count5 / valid5_count) * 100.0
        else:
            coverage5 = np.nan
        
        difference = coverage5 - coverage4
        
        results.append({
            'time': time_val,
            'fmask4_coverage': coverage4,
            'fmask5_coverage': coverage5,
            'difference': difference
        })
    
    df = pd.DataFrame(results)
    
    if verbose and len(df) > 0:
        mask_name = "Cloud" if mask_value == 4 else "Cloud Shadow" if mask_value == 2 else f"Mask {mask_value}"
        print(f"{mask_name} Coverage Comparison:")
        print(f"  Mean Fmask 4.7 coverage: {df['fmask4_coverage'].mean():.2f}%")
        print(f"  Mean Fmask 5.0 coverage: {df['fmask5_coverage'].mean():.2f}%")
        print(f"  Mean difference (Fmask 5.0 - 4.7): {df['difference'].mean():.2f}%")
    
    return df


In [ ]:
def classify_mrgs_ids_by_climate_zone(df_fmask_percentage, defined_zones=None, verbose=True):
    """
    Classify MRGS IDs by climate zone and add a 'climate_zone' column to df_fmask_percentage.
    
    This function classifies each unique MRGS ID based on its geographic location using
    the Köppen-Geiger climate classification. Only defined climate zones are kept;
    undefined zones are set to None.
    
    Parameters
    ----------
    df_fmask_percentage : pandas.DataFrame
        DataFrame containing Fmask comparison data. Must have a 'mrgs_id' column.
    defined_zones : list, optional
        List of climate zones to keep. Default is ['Arid', 'Cold', 'Temperate'].
        Any MRGS IDs classified to other zones will be set to None.
    verbose : bool, optional
        If True, print summary statistics. Default is True.
    
    Returns
    -------
    pandas.DataFrame
        The input DataFrame with an added 'climate_zone' column.
    
    Notes
    -----
    - The function uses the `classify_climate_zone()` function from the utilities module.
    - MRGS IDs that fail classification (e.g., invalid MGRS conversion) are set to None.
    - The function is efficient: it classifies unique MRGS IDs once, then maps to all rows.
    
    Examples
    --------
    >>> df_fmask_percentage = classify_mrgs_ids_by_climate_zone(df_fmask_percentage)
    >>> # Filter by climate zone
    >>> cold_data = df_fmask_percentage[df_fmask_percentage['climate_zone'] == 'Cold']
    """
    import mgrs
    
    if defined_zones is None:
        defined_zones = ['Arid', 'Cold', 'Temperate']
    
    if verbose:
        print("="*80)
        print("CLIMATE ZONE CLASSIFICATION FOR df_fmask_percentage")
        print("="*80)
    
    # Check if climate_zone column already exists
    if 'climate_zone' in df_fmask_percentage.columns:
        if verbose:
            print("⚠️  Warning: 'climate_zone' column already exists. Overwriting...")
        df_fmask_percentage = df_fmask_percentage.drop(columns=['climate_zone'])
    
    # Get unique MRGS IDs to classify (more efficient than classifying each row)
    unique_mrgs_ids = df_fmask_percentage['mrgs_id'].unique()
    if verbose:
        print(f"\nClassifying {len(unique_mrgs_ids)} unique MRGS IDs...")
    
    # Create a mapping dictionary: mrgs_id -> climate_zone
    m = mgrs.MGRS()
    mrgs_to_climate_zone = {}
    undefined_count = 0
    
    for mrgs_id in unique_mrgs_ids:
        mrgs_str = str(mrgs_id).strip()
        mgrs_str = mrgs_str[1:] if mrgs_str.startswith('T') else mrgs_str
        
        try:
            lat, lon = m.toLatLon(f"{mgrs_str} 50000 50000")
            climate_zone = classify_climate_zone(lat, lon)
            
            # Only keep defined climate zones, set others to None
            if climate_zone in defined_zones:
                mrgs_to_climate_zone[mrgs_id] = climate_zone
            else:
                mrgs_to_climate_zone[mrgs_id] = None
                undefined_count += 1
        except Exception as e:
            # Failed classification -> None
            mrgs_to_climate_zone[mrgs_id] = None
    
    # Map climate zones to df_fmask_percentage
    df_fmask_percentage['climate_zone'] = df_fmask_percentage['mrgs_id'].map(mrgs_to_climate_zone)
    
    if verbose:
        # Print summary statistics
        print(f"\n✓ Climate zone classification complete!")
        print(f"\nSummary:")
        print(f"  Total granules: {len(df_fmask_percentage)}")
        print(f"  Unique MRGS IDs: {len(unique_mrgs_ids)}")
        print(f"  MRGS IDs with undefined zones: {undefined_count}")
        
        # Count by climate zone
        print(f"\nClimate Zone Distribution:")
        for zone in defined_zones:
            count = len(df_fmask_percentage[df_fmask_percentage['climate_zone'] == zone])
            pct = 100 * count / len(df_fmask_percentage) if len(df_fmask_percentage) > 0 else 0
            print(f"  {zone}: {count} granules ({pct:.1f}%)")
        
        # Count None/undefined
        none_count = df_fmask_percentage['climate_zone'].isna().sum()
        if none_count > 0:
            pct = 100 * none_count / len(df_fmask_percentage)
            print(f"  Undefined/None: {none_count} granules ({pct:.1f}%)")
        
        print(f"\n✓ 'climate_zone' column added to df_fmask_percentage")
        print("="*80)
    
    return df_fmask_percentage

